# LLM4ChipDesign HW1 Chipchat
## ExampleA (Example 1 - binary_to_bcd (combinational converter))
Heng Pu   
hp2723

### 1. Setting up


In [1]:
### Installing dependencies
!pip install openai

!apt-get update
!apt-get install -y iverilog

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:5 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,608 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,009 kB]
Fetched 5,745 kB in 2s (2,792 kB/s)
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (so

In [6]:
def ask_deepseek(prompt):
    from openai import OpenAI
    client = OpenAI(
        api_key = "sk-4b4b50b8153147759b5530268e51a1f7",
        base_url = "https://api.deepseek.com"
    )
    completion = client.chat.completions.create(
        model="deepseek-reasoner",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=2048,
        temperature=0.7
    )
    return completion.choices[0].message.content

### 2. binary_to_bcd (combinational converter)

In [7]:
! mkdir -p binary_to_bcd
! cd binary_to_bcd && curl -O https://raw.githubusercontent.com/FCHXWH823/LLM4ChipDesign/fe806e8f8b7cb8442ce161f452d070cfcf953656/VerilogGenBenchmark/TestBench/binary_to_bcd_tb.v

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1078  100  1078    0     0   3413      0 --:--:-- --:--:-- --:--:--  3422


In [8]:
verilog_generation_prompt = '''
I am trying to create a Verilog model binary_to_bcd_converter for a binary to binary-coded-decimal converter. It must meet the following specifications:
	- Inputs:
		- Binary input (5-bits)
	- Outputs:
		- BCD (8-bits: 4-bits for the 10's place and 4-bits for the 1's place)

How would I write a design that meets these specifications?
'''

In [9]:
ask_deepseek(verilog_generation_prompt)

"To create a Verilog model for a 5-bit binary to BCD converter, you can follow this approach:\n\n### Approach\nSince the input is 5 bits (range 0–31), the BCD output will have two digits (tens and ones), each represented by 4 bits. The converter can be implemented using combinational logic or a lookup table (LUT). A straightforward method is to use a case statement that maps each 5-bit binary value to its corresponding 8-bit BCD representation.\n\n### Steps\n1. **Define the module** with a 5-bit input (`binary_in`) and an 8-bit output (`bcd_out`), where:\n   - `bcd_out[7:4]` represents the tens digit.\n   - `bcd_out[3:0]` represents the ones digit.\n2. **Use a case statement** to map all 32 possible binary inputs (0–31) to their BCD equivalents.\n3. **Handle invalid inputs** (if any) by defaulting to a known value (e.g., 0).\n\n### Verilog Code\n```verilog\nmodule binary_to_bcd_converter (\n    input [4:0] binary_in,  // 5-bit binary input\n    output reg [7:0] bcd_out // 8-bit BCD out

In [10]:
output_verilog_code = '''
module binary_to_bcd_converter (
    input wire [4:0] binary_input, // 5-bit binary input
    output reg [7:0] bcd_output     // 8-bit BCD output (4 bits for tens, 4 bits for ones)
);

// Internal registers for BCD representation
reg [3:0] bcd_tens; // Tens place
reg [3:0] bcd_ones; // Ones place
integer i; // Loop variable

always @* begin
    // Initialize BCD values to 0
    bcd_tens = 4'b0000;
    bcd_ones = 4'b0000;

    // Convert binary to BCD using double-dabble algorithm
    for (i = 0; i < 5; i = i + 1) begin
        // Shift left the BCD digits to make room for the next binary bit
        if (bcd_tens >= 5)
            bcd_tens = bcd_tens + 4'b0011; // Add 3 to tens place if necessary

        if (bcd_ones >= 5)
            bcd_ones = bcd_ones + 4'b0011; // Add 3 to ones place if necessary

        // Shift BCD values left to add a new binary bit
        {bcd_tens, bcd_ones} = {bcd_tens, bcd_ones} << 1;

        // Add the next bit of the binary input
        bcd_ones[0] = binary_input[4 - i]; // Add the next bit of the binary input
    end

    // Combine BCD places into the final output
    bcd_output = {bcd_tens, bcd_ones};
end

endmodule
'''
filename = "binary_to_bcd/binary_to_bcd.v"
# Write the extracted Verilog code to the file
with open(filename, "w") as f:
    f.write(output_verilog_code)

In [11]:
!cd binary_to_bcd/ && iverilog -g2012 -o binary_to_bcd.vvp binary_to_bcd.v binary_to_bcd_tb.v && vvp binary_to_bcd.vvp

Testing Binary-to-BCD Converter...
VCD info: dumpfile my_design.vcd opened for output.
All test cases passed!
